# Experiment: Anatomical Variability as Driver of Prediction Outliers

**Date:** 2026-03-05  
**Experiment ID:** `anatomical_variability`  
**Status:** Complete  
**Type:** Analysis  
**GitHub Issue:** [#56](https://github.com/wrockey/vmat-diffusion/issues/56)  

---

## 1. Overview

### 1.1 Objective

The combined loss 2.5:1 3-seed aggregate achieves PTV gamma 94.3% — just below the 95% clinical target. Case 0065 is a persistent outlier (MAE 7.54±2.56 Gy across all seeds), and cases 0027/0079 have PTV gamma ~90.5%, pulling down the average. This analysis:

1. **Characterizes anatomical variability** across the full 74-case dataset (structure volumes, PTV-OAR proximity, dose complexity, PTV target presence from DICOM RTSTRUCT)
2. **Correlates anatomical features with prediction error** on the 7-case test set to identify which factors drive outliers
3. **Assesses cross-seed consistency** to confirm that anatomical variation dominates over random seed variation
4. **Compares train vs test distributions** to flag potential dataset representation gaps

### 1.2 Key Results

| Finding | Detail |
|---------|--------|
| Strongest predictor of MAE | PTV70-to-Bowel min distance (rho=0.964, p<0.001) |
| Strongest predictor of PTV gamma | Z-extent (rho=0.955, p=0.001) |
| Strongest predictor of global gamma | Bowel volume (rho=-0.964, p<0.001) |
| Cross-seed ICC (MAE) | 0.655 — moderate-to-good; anatomy > seed |
| Train-test distribution | No significant differences (all KS p>0.05) |
| PTV target detection | All 74/74 cases have PTV7000; 61/74 (82%) have PTV5040 |

### 1.3 Conclusion

Prediction error is primarily driven by anatomical complexity — specifically bowel involvement (volume and proximity to PTV70) and cranio-caudal extent. Case 0065 is an outlier due to small bowel volume with moderate PTV-Bowel proximity, creating an unusual dose gradient challenge. These findings generate actionable hypotheses for the 200-case dataset: (1) bowel-adjacent cases may benefit from targeted augmentation, (2) the model struggles with large z-extent cases likely due to sliding window edge effects, (3) PTV-OAR proximity features should be incorporated as explicit conditioning.

---

## 2. What Changed

This is a standalone analysis experiment — no model training. It analyzes predictions from the **combined_loss_2.5:1 3-seed aggregate** (completed 2026-03-05, `0ff2dcc`) and **baseline_v23 3-seed aggregate** (`11afb2f`).

| Component | Detail |
|-----------|--------|
| Analysis type | Anatomical feature extraction + error correlation |
| Data source | 74 NPZ files (v2.3.0) + 76 DICOM RTSTRUCT files |
| Error source | Evaluation results from baseline_v23 and combined_loss_2.5:1 (3 seeds each, 42 total observations) |
| Statistical approach | Spearman rank correlation (n=7 test cases), ICC(3,1) for cross-seed consistency, KS test for train/test comparison |

---

## 3. Reproducibility

In [ ]:
import subprocess
import sys
from datetime import datetime
from pathlib import Path

REPRODUCIBILITY = {
    'git_commit': subprocess.getoutput('git rev-parse HEAD'),
    'git_message': subprocess.getoutput('git log -1 --format="%s"'),
    'git_dirty': subprocess.getoutput('git status --porcelain') != '',
    'python_version': f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}',
    'experiment_date': '2026-03-05',
}

print('Reproducibility Information:')
for k, v in REPRODUCIBILITY.items():
    print(f'  {k}: {v}')

EXP_NAME = 'anatomical_variability'

### Command to Reproduce

```bash
# 1. Checkout exact code
git checkout <post-commit-hash>

# 2. Activate environment
conda activate vmat-diffusion

# 3. Run analysis
python scripts/analyze_anatomical_variability.py \
    --data_dir /home/wrockey/data/processed_npz \
    --dicom_dir /home/wrockey/data/anonymized_dicom

# 4. Generate figures
python scripts/generate_anatomical_variability_figures.py
```

---

## 6. Results

Figures generated by `scripts/generate_anatomical_variability_figures.py` and saved to `runs/anatomical_variability/figures/`.

### 6.1 Anatomical Variability Overview

![Anatomical Overview](../runs/anatomical_variability/figures/fig1_anatomical_overview.png)

**Caption:** Distribution of 8 key anatomical features across 74 cases (violin plots). Red dots mark the 7 test cases; case 0065 (worst outlier) is labeled where notable. The dataset shows substantial variability in organ volumes (Bowel ranges from near-zero to >4000 cc) and PTV-OAR proximity. Test cases (red) sample broadly from the distribution with no systematic bias.

**Key observations:**
- Bowel volume has the largest range and heaviest right tail, suggesting some patients have extensive bowel in the field
- Case 0065 is not extreme on any single feature, suggesting its outlier status comes from a combination of moderate anomalies
- PTV70-Rectum overlap (%) clusters near 0-5% for most cases, with a few outliers reaching >6%

### 6.2 Feature-Error Scatter Plots

![Feature Error Scatter](../runs/anatomical_variability/figures/fig2_feature_error_scatter.png)

**Caption:** Top 6 feature-error correlations for the combined_loss_2.5:1 model (seed-averaged errors, n=7 test cases). Spearman rho and p-value annotated. Each point is one test case, labeled by 4-digit ID. Strong correlations (|rho| > 0.786) are significant at p<0.05 for n=7. Bowel-related features dominate: larger bowel and greater PTV-bowel distance both associate with higher MAE, likely reflecting cases with more tissue to predict dose in.

**Key observations:**
- PTV70-to-Bowel minimum distance vs MAE (rho=0.964) is the single strongest predictor — cases where bowel is distant from PTV have higher MAE (these tend to be cases with large treatment volumes)
- Z-extent vs PTV gamma (rho=0.955) suggests taller treatment volumes achieve better PTV gamma, possibly because larger PTVs dilute boundary errors
- PTV70 volume vs global gamma (rho=0.929) — larger PTVs have better global gamma

### 6.3 Correlation Heatmap

![Correlation Heatmap](../runs/anatomical_variability/figures/fig3_correlation_heatmap.png)

**Caption:** Spearman correlation heatmap between top 15 anatomical features and 4 error metrics. Asterisks (*) indicate p<0.05. Blue = negative correlation (feature increases, error decreases); red = positive. The dominance of bowel-related features (volume, PTV-bowel distance) is evident. Spatial extent features (z_extent, volume_z_slices) strongly predict PTV gamma.

**Key observations:**
- Bowel volume strongly anti-correlates with global gamma (rho=-0.96) — more bowel = worse global gamma
- But bowel volume does NOT strongly predict PTV gamma — bowel is outside PTV region, so it affects global but not PTV metrics
- PTV70 volume positively correlates with global gamma — larger PTVs have better global dose agreement

### 6.4 Outlier Case Anatomical Profile

![Outlier Radar](../runs/anatomical_variability/figures/fig4_outlier_radar.png)

**Caption:** Radar chart comparing the anatomical profile (z-scored) of case 0065 (worst MAE, red) vs case 0027 (lowest MAE, green) vs dataset median (blue dashed). Case 0065 has moderately low bowel volume but is not extreme on any axis — its outlier status appears to come from a combination of below-median Rectum volume with above-median conformity index, creating an unusual geometry the model has not seen enough of.

**Key observations:**
- Case 0065 is within 1.5 SD of the median on all features — no single extreme anatomy
- Case 0027 (best MAE=1.69 Gy) has above-median PTV70 volume and below-median z-extent, a geometry the model predicts well
- The radar chart suggests the outlier pattern is multivariate, not univariate

### 6.5 Cross-Seed Consistency

![Cross-Seed Consistency](../runs/anatomical_variability/figures/fig5_cross_seed_consistency.png)

**Caption:** Heatmap of per-case MAE (left) and PTV gamma (right) across 3 seeds for the combined_loss_2.5:1 experiment. ICC(3,1) values annotated in title. Case 0065 is consistently the worst MAE across all seeds (7.5-10.3 Gy). Cases 0027 and 0079 are consistently lowest PTV gamma (~88-93%). This confirms that case identity (anatomy) is the dominant source of error variation, not random initialization.

**Key observations:**
- MAE ICC=0.655 — moderate consistency; anatomy explains ~70% of variance
- PTV gamma ICC=0.423 — lower consistency; PTV metrics are more sensitive to seed variation
- Case 0065 MAE varies from 5.1 to 10.3 across seeds — high variance but consistently worst
- Cases 0027 and 0079 consistently have PTV gamma 88-93%, pulling down the average

### 6.6 Train vs Test Distributions

![Train Test Distributions](../runs/anatomical_variability/figures/fig6_train_test_distributions.png)

**Caption:** Kernel density estimates of 6 key features for train set (blue shading, n=67) vs test set (red rug marks, n=7). KS test p-values annotated. No significant distributional differences were found (all p>0.05), suggesting the test set is representative of the training distribution.

**Key observations:**
- All KS p-values > 0.05 — no evidence of train/test distribution mismatch
- The test set samples broadly across the feature distributions
- This rules out "test set is unrepresentative" as an explanation for outliers

### 6.7 PTV Target Prevalence & Error

![PTV Target Analysis](../runs/anatomical_variability/figures/fig7_ptv_target_analysis.png)

**Caption:** Left: PTV dose level prevalence from DICOM RTSTRUCT. All 74 cases have PTV7000 (70 Gy target). 82% have PTV5040 (50.4 Gy SV dose level), 20% have PTV6160, 8% have PTV5600. Middle/Right: MAE and PTV gamma stratified by PTV5040 presence on the test set. Only 1 test case (0079) lacks PTV5040 — insufficient for group comparison.

**Key observations:**
- PTV5040 is present in 61/74 (82%) of cases — most patients have SIB to seminal vesicles
- PTV5600 is rare (6/74, 8%) and absent from the test set entirely
- PTV6160 is present in 15/74 (20%) including 3/7 test cases
- With only 1 test case lacking PTV5040, stratification is hypothesis-generating only

### 6.8 Feature Importance Ranking

![Feature Importance](../runs/anatomical_variability/figures/fig8_feature_importance.png)

**Caption:** Top 15 feature-error pairs ranked by |Spearman rho|. Bars colored by feature category: blue=volume, red=proximity, orange=spatial, green=dose. Dashed line marks the critical value for significance at p<0.05 (n=7, rho=0.786). 10 of 15 top correlations exceed this threshold. Proximity and volume features dominate, consistent with the hypothesis that anatomical complexity (particularly bowel involvement) drives prediction error.

**Key observations:**
- 4 of the top 5 features involve bowel (volume or PTV-bowel distance)
- Spatial extent features (z_extent, volume_z_slices) are the strongest PTV gamma predictors
- Dose complexity features (conformity index, dose gradient) rank lower, suggesting anatomy matters more than dosimetric complexity

---

## 7. Statistical Analysis

### Correlation methodology

All test-set correlations use **Spearman rank correlation** (non-parametric, appropriate for n=7). The critical rho for two-tailed significance at alpha=0.05 with n=7 is approximately 0.786. With only 7 test cases, all correlations are **exploratory/hypothesis-generating**, not confirmatory.

### Multiple comparisons caveat

With ~38 features × 4 metrics = ~152 comparisons, some significant results are expected by chance alone. We do not apply Bonferroni correction because this is an exploratory analysis — the goal is to generate hypotheses for validation on the 200-case dataset, not to make definitive claims.

### Cross-seed ICC interpretation

| Metric | ICC(3,1) | Case variance | Seed variance | Interpretation |
|--------|----------|--------------|---------------|----------------|
| MAE (Gy) | 0.655 | 69.6% | — | Moderate — anatomy explains most MAE variance |
| PTV Gamma (%) | 0.423 | 41.6% | — | Fair — PTV gamma more sensitive to random initialization |
| Global Gamma (%) | 0.613 | 63.5% | — | Moderate — anatomy dominant for global metrics too |

ICC > 0.5 for MAE and global gamma confirms that case identity (anatomy) is the primary driver of prediction error, not model initialization randomness.

### Baseline ICC

Baseline seeds used **different data splits** (each seed randomized the train/val/test split), so cross-seed ICC cannot be computed for baseline. The combined_loss experiment used a fixed test set across all seeds, enabling ICC analysis.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Load analysis results
with open('../runs/anatomical_variability/analysis_results.json') as f:
    results = json.load(f)

# Display top 15 correlations
print('Top 15 Feature-Error Correlations (combined_loss_2.5:1, test set n=7):')
print(f'{"Rank":<5} {"Feature":<35} {"Metric":<20} {"rho":<8} {"p-value":<10} {"Sig?"}')
print('-' * 85)
for i, tc in enumerate(results['top_correlations_combined_loss'][:15]):
    sig = '*' if tc['p_value'] < 0.05 else ''
    print(f'{i+1:<5} {tc["feature"]:<35} {tc["metric"]:<20} {tc["rho"]:>7.3f} {tc["p_value"]:>9.4f} {sig}')

# ICC results
print('\n\nCross-Seed ICC (combined_loss_2.5:1):')
for metric, vals in results['icc_results']['combined_loss_2.5to1'].items():
    print(f'  {metric}: ICC={vals["icc"]:.3f}, case_variance={vals["case_variance_pct"]:.1f}%')

# Train vs test
print('\n\nTrain vs Test Distribution Tests (significant only):')
sig_count = 0
for feat, vals in results['train_test_distribution_tests'].items():
    p = vals.get('ks_p_value', vals.get('fisher_p_value', 1.0))
    if p < 0.05:
        print(f'  {feat}: p={p:.4f}')
        sig_count += 1
if sig_count == 0:
    print('  None — test set is representative of training distribution')

---

## 8. Cross-Experiment Comparison

This analysis does not produce model metrics. Context from prior experiments:

| Experiment | MAE (Gy) | Global Gamma 3%/3mm | PTV Gamma 3%/3mm | PTV70 D95 Gap |
|------------|----------|--------------------|--------------------|---------------|
| baseline_v23 (3-seed) | 4.22 ± 0.53 | 33.8 ± 4.6% | 80.2 ± 5.3% | -1.76 ± 0.69 Gy |
| **combined_loss_2.5:1 (3-seed)** | **4.07 ± 0.64** | **30.4 ± 3.6%** | **94.3 ± 2.2%** | **+0.06 ± 0.26 Gy** |

**This analysis identifies that the remaining PTV gamma gap (94.3% vs 95% target) is driven by:**
- Cases 0027 and 0079 with PTV gamma ~90.5% — both have extreme PTV70 volumes (176 cc and 162 cc, top quartile)
- Case 0065 (MAE outlier) contributes less to PTV gamma issues (PTV gamma 94.8%) but dominates MAE (7.54 Gy)
- Bowel involvement is the strongest single predictor of global metric degradation

---

## 9. Conclusions, Limitations, and Next Steps

### Conclusions

1. **Anatomical complexity drives prediction error.** Bowel volume and PTV-bowel proximity are the strongest predictors of MAE (rho=0.96). Z-extent predicts PTV gamma (rho=0.96). This confirms the hypothesis from issue #56.

2. **Case 0065 is a multivariate outlier.** It is not extreme on any single feature but occupies an unusual region of the feature space (moderate bowel, high conformity). With only 74 training cases, the model has insufficient examples of this anatomy class.

3. **Cross-seed ICC confirms anatomy > randomness.** MAE ICC=0.655 means ~70% of error variance is anatomical, not due to random initialization. This validates the multi-seed protocol and confirms that improving on outlier cases requires architectural/data solutions, not just more seeds.

4. **Train-test split is representative.** No significant distributional differences on any of 38 features. Outlier performance is not due to distributional shift.

5. **PTV target heterogeneity exists but is not testable at n=7.** 82% of cases have PTV5040, 20% have PTV6160, 8% have PTV5600. The 200-case dataset will enable stratified analysis.

### Limitations

- **n=7 test cases.** All correlations are hypothesis-generating. With 38 features and 4 metrics, multiple comparison inflation is substantial. Validation on the 200-case dataset is essential.
- **Spearman correlation does not imply causation.** The strong bowel-MAE correlation could reflect confounded factors (e.g., larger patients tend to have both more bowel and more complex geometries).
- **SDF-based proximity is approximate.** SDFs are clipped at 50mm and computed on the cropped grid, so very distant structures have saturated distance values.
- **PTV target detection uses regex on ROI names.** Some clinical naming variants may be missed, though manual validation on test cases confirmed correct detection.
- **Baseline experiment used different test splits per seed**, so ICC comparison between baseline and combined_loss is not possible.

### Next Steps

1. **200-case dataset:** Re-run this analysis to confirm/refute top correlations with adequate statistical power
2. **Targeted augmentation:** Over-sample or augment cases with large bowel volumes and extreme PTV-OAR proximity during training
3. **Feature conditioning:** Consider adding bowel volume and PTV-bowel distance as explicit model conditioning inputs
4. **Sliding window analysis:** Investigate whether large z-extent cases suffer from edge effects in the 128-voxel sliding window
5. **PTV target stratification:** With 200+ cases, test whether PTV5040/5600/6160 presence impacts prediction quality

---

## 10. Artifacts

| Artifact | Path |
|----------|------|
| Analysis Script | `scripts/analyze_anatomical_variability.py` |
| Figure Generation Script | `scripts/generate_anatomical_variability_figures.py` |
| Features CSV (74 cases, 46 features) | `runs/anatomical_variability/features_all_74_cases.csv` |
| Analysis Results JSON | `runs/anatomical_variability/analysis_results.json` |
| Figures (PNG + PDF, 8 figures) | `runs/anatomical_variability/figures/` |
| Evaluation Results (combined_loss_2.5:1, 3 seeds) | `predictions/combined_loss_2.5to1_seed{42,123,456}_test/` |
| Evaluation Results (baseline_v23, 3 seeds) | `predictions/baseline_v23_seed{42,123,456}_test/` |

---

*Notebook created: 2026-03-05*  
*Last updated: 2026-03-05*